Tutorial 3: Inversions
======================

In the previous two tutorials, we introduced:

 - `Pixelization`'s: which place a pixel-grid in the source-plane.
 - `Mappers`'s: which describe how each source-pixel maps to one or more image pixels.

However, none of this has actually helped us fit strong lens data or reconstruct the source galaxy. This is the subject
of this tutorial, where the process of reconstructing the source's light on the pixelization is called an `Inversion`.

__Contents__

- **Initial Setup:** we'll use the same strong lensing data as the previous tutorial, where.
- **Pixelization:** Finally, we can now use the `Mapper` to reconstruct the source via an `Inversion`.
- **Positive Only Solver:** Ensuring positive-only solutions for linear light profile intensities.
- **Wrap Up:** Summary of the script and next steps.
- **Detailed Explanation:** If you are interested in a more detailed description of how inversions work, then checkout the file.

__Google Colab Setup__

Set up Google Colab as in [the introductory notebook](https://github.com/PyAutoLabs/HowToLens/blob/main/notebooks/chapter_1_introduction/tutorial_1_grids_and_galaxies.ipynb).


In [ ]:
# @title Set up Google Colab { display-mode: "form" }
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("howtolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import autolens as al
import autolens.plot as aplt
import autoarray.plot as aaplt

__Initial Setup__

we'll use the same strong lensing data as the previous tutorial, where:

 - The lens galaxy's light is omitted.
 - The lens system uses an `Isothermal` galaxy mass profile and a separate `ExternalShear` field.
 - The source galaxy's light is an `Sersic`.

In [ ]:
dataset_name = "simple__no_lens_light"
dataset_path = Path("dataset") / "imaging" / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/simulator/no_lens_light.py"],
        check=True,
    )

dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    psf_path=dataset_path / "psf.fits",
    pixel_scales=0.1,
)

Lets create an annular mask which traces the strongly lensed source's ring of light, and apply it to the dataset.

In [ ]:
mask = al.Mask2D.circular_annular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    inner_radius=0.5,
    outer_radius=2.8,
)

dataset = dataset.apply_mask(mask=mask)

Now the mask is applied, plotting the data draws it over the image, letting us check the annulus traces the ring of
lensed source light.

In [ ]:
aplt.plot_array(array=dataset.data, title="Image Data With Mask Applied")

We now create the masked source-plane grid via the tracer, as we did in the previous tutorial.

In [ ]:
lens_galaxy = al.Galaxy(
    redshift=0.5,
    mass=al.mp.Isothermal(
        centre=(0.0, 0.0),
        einstein_radius=1.6,
        ell_comps=al.convert.ell_comps_from(axis_ratio=0.9, angle=45.0),
    ),
)
field = al.MassField(
    redshift=0.5, shear=al.mp.ExternalShear(gamma_1=0.05, gamma_2=0.05)
)

tracer = al.Tracer(galaxies=[lens_galaxy, al.Galaxy(redshift=1.0)], fields=[field])

source_plane_grid = tracer.traced_grid_2d_list_from(grid=dataset.grids.pixelization)[1]

we again use the rectangular pixelization to create the mapper.

(Ignore the regularization input below for now, we will cover this in the next tutorial).

In [ ]:
mesh = al.mesh.RectangularBilinearAdaptDensity(shape=(25, 25))

pixelization = al.Pixelization(mesh=mesh)

interpolator = mesh.interpolator_from(
    source_plane_data_grid=source_plane_grid,
    source_plane_mesh_grid=None,
)

mapper = al.Mapper(
    interpolator=interpolator,
    regularization=al.reg.Constant(coefficient=1.0),
)


aplt.plot_grid(grid=mapper.source_plane_mesh_grid, title="Source-Plane Mesh Grid")

__Pixelization__

Finally, we can now use the `Mapper` to reconstruct the source via an `Inversion`. I'll explain how this works in a 
second, but lets just go ahead and create the inversion first. 

In [ ]:
inversion = al.Inversion(dataset=dataset, linear_obj_list=[mapper])

The inversion has reconstructed the source's light on the rectangular pixel grid, which is called the 
`reconstruction`. This source-plane reconstruction can be mapped back to the image-plane to produce the 
`mapped_reconstructed_operated_data`.

In [ ]:
print(inversion.reconstruction)
print(inversion.mapped_reconstructed_operated_data)

Both of these can be plotted using an `aplt.plot_array`.

It is possible for an inversion to have multiple `Mapper`'s, therefore for certain figures we specify the index 
of the mapper we wish to plot. In this case, because we only have one mapper we specify the index 0.

In [ ]:
aplt.plot_array(
    array=inversion.mapped_reconstructed_operated_data, title="Inversion Reconstruction"
)

There we have it, we have successfully reconstructed the source using a rectangular pixel-grid. Whilst this source 
was simple (a blob of light in the centre of the source-plane), inversions come into their own when fitting sources 
with complex morphologies. 

Lets use an inversion to reconstruct a complex source!

In [ ]:
dataset_name = "source_complex"
dataset_path = Path("dataset") / "imaging" / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/simulator/source_complex.py"],
        check=True,
    )

dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    psf_path=dataset_path / "psf.fits",
    pixel_scales=0.05,
)

aplt.plot_array(array=dataset.data, title="Data")

This code is doing all the same as above (setup the mask, galaxy, tracers, mapper, inversion, etc.).

In [ ]:
mask = al.Mask2D.circular_annular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    inner_radius=0.1,
    outer_radius=3.2,
)

dataset = dataset.apply_mask(mask=mask)

aplt.plot_array(array=dataset.data, title="Image Data With Mask Applied")

lens_galaxy = al.Galaxy(
    redshift=0.5,
    mass=al.mp.Isothermal(
        centre=(0.0, 0.0), einstein_radius=1.6, ell_comps=(0.17647, 0.0)
    ),
)

tracer = al.Tracer(galaxies=[lens_galaxy, al.Galaxy(redshift=1.0)], fields=[field])

source_plane_grid = tracer.traced_grid_2d_list_from(grid=dataset.grids.pixelization)[1]

interpolator = mesh.interpolator_from(
    source_plane_data_grid=source_plane_grid,
    source_plane_mesh_grid=None,
)

mapper = al.Mapper(
    interpolator=interpolator,
    regularization=al.reg.Constant(coefficient=1.0),
)

inversion = al.Inversion(dataset=dataset, linear_obj_list=[mapper])

Now lets plot the complex source reconstruction.

In [ ]:
aplt.plot_array(
    array=inversion.mapped_reconstructed_operated_data, title="Inversion Reconstruction"
)

Pretty great, huh? If you fitted complex sources using light profiles in the search chaining tutorials at the end of
chapter 2 (tutorials 9-11), you'll remember that getting a model image that looked this good simply *was not possible*.
With an inversion, we can do this with ease and without having to perform model-fitting with 20+ parameters for the
source's light!

We will now briefly discuss how an inversion actually works, however the explanation I give in this tutorial will be 
overly-simplified. To be good at lens modeling you do not need to understand the details of how an inversion works, you 
simply need to be able to use an inversion to model a strong lens. 

To begin, lets consider some random mappings between our mapper`s source-pixels and the image.

`mapper.mappings_from` returns one `Mapping` per group of source-pixels, which we pass straight to
`subplot_image_and_mapper` as `regions=`. Each mapping is drawn in its own colour: the source-plane cell on the right,
and the image-plane regions it maps to on the left. Tutorial 2 covers this in detail.

In [ ]:
pix_indexes = [[min(index, mapper.pixels - 1)] for index in [445, 285, 313, 132, 211]]

mappings = mapper.mappings_from(pix_indexes=pix_indexes)

aaplt.subplot_image_and_mapper(
    mapper=mapper,
    image=dataset.data,
    regions=mappings,
    mesh_grid=mapper.source_plane_mesh_grid,
)

These mappings are known before the inversion reconstructs the source galaxy, which means before this inversion is
performed we know two key pieces of information:

 1) The mappings between every source-pixel and sets of image-pixels.
 2) The flux values in every observed image-pixel, which are the values we want to fit successfully.

It turns out that with these two pieces of information we can linearly solve for the set of source-pixel fluxes that
best-fit (e.g. maximize the log likelihood) our observed image. Essentially, we set up the mappings between source and
image pixels as a large matrix and solve for the source-pixel fluxes in an analogous fashion to how you would solve a
set of simultaneous linear equations. This process is called a `linear inversion`.

In the language of linear algebra: the mappings form a `mapping matrix` $f$, which together with the data and
noise-map defines a linear system whose solution is the vector of source-pixel fluxes $s$. Tutorial 5 of this
chapter writes this system out in full and computes it step-by-step in code -- for now, the intuition above is all
you need.

There are three more things about a linear inversion that are worth knowing:

 1) When performing fits using light profiles, we discussed how a `model_image` was generated by convolving the light
 profile images with the data's PSF. A similar blurring operation is incorporated into the inversion, such that it 
 reconstructs a source (and therefore image) which fully accounts for the telescope optics and effect of the PSF.

 2) You may be familiar with image sub-gridding, which splits each image-pixel into a sub-pixel (if you are not 
 familiar then feel free to checkout the optional **HowToLens** tutorial on sub-gridding. If a sub-grid is used, it is 
 the mapping between every sub-pixel and source-pixel that is computed and used to perform the inversion. This prevents 
 aliasing effects degrading the image reconstruction. By default **PyAutoLens** uses sub-gridding of degree 4x4.

 3) The inversion's solution is regularized. But wait, that's what we'll cover in the next tutorial!

Finally, let me show you how easy it is to fit an image with an `Inversion` using a `FitImaging` object. Instead of 
giving the source galaxy a light profile, we simply pass it a `Pixelization` and regularization, and pass it to a 
tracer.

In [ ]:
# `PYAUTO_SMALL_DATASETS=1` (the CI smoke harness) caps the data to a 16 x 16 image, where a
# 40 x 40 mesh reconstructs 1600 source pixels from a few dozen image pixels. The mesh is
# capped alongside the data; a normal full-resolution run keeps the 40 x 40 grid described above.
mesh_shape = (
    al.util.dataset.SMALL_DATASETS_SHAPE_NATIVE
    if dataset.shape_native == al.util.dataset.SMALL_DATASETS_SHAPE_NATIVE
    else (40, 40)
)

pixelization = al.Pixelization(
    mesh=al.mesh.RectangularBilinearAdaptDensity(shape=mesh_shape),
    regularization=al.reg.Constant(coefficient=1.0),
)

source_galaxy = al.Galaxy(redshift=1.0, pixelization=pixelization)

tracer = al.Tracer(galaxies=[lens_galaxy, source_galaxy], fields=[field])

Then, like before, we pass the imaging and tracer `FitImaging` object. 

We see some pretty good looking residuals, we must be fitting the lensed source accurately! In fact, we can use the
`subplot_of_planes` method to specifically visualize the inversion and plot the source reconstruction.

In [ ]:
fit = al.FitImaging(dataset=dataset, tracer=tracer)

aplt.subplot_fit_imaging(fit=fit)
aplt.subplot_fit_imaging_of_planes(fit=fit)

__Positive Only Solver__

All pixelized source reconstructions use a positive-only solver, meaning that every source-pixel is only allowed
to reconstruct positive flux values. This ensures that the source reconstruction is physical and that we don't
reconstruct negative flux values that don't exist in the real source galaxy (a common systematic solution in lens
analysis).

It may be surprising to hear that this is a feature worth pointing out, but it turns out setting up the linear algebra
to enforce positive reconstructions is difficult to make efficient. A lot of development time went into making this
possible, where a bespoke fast non-negative linear solver was developed to achieve this.

Other methods in the literature often do not use a positive only solver, and therefore suffer from these 
unphysical solutions, which can degrade the results of lens model in general.

__Wrap Up__

And, we're done, here are a few questions to get you thinking about inversions:

 1) The inversion provides the maximum log likelihood solution to the observed image. Is there a problem with seeking 
 the highest likelihood solution? Is there a risk that we're going to fit other things in the image than just the 
 lensed source galaxy? What happens if you reduce the `coefficient` of the regularization object above to zero?

 2) The exterior pixels in the rectangular pixel-grid have no image-pixels in them. However, they are still given a 
 reconstructed flux. Given these pixels do not map to the data, where is this value coming from?
 
__Detailed Explanation__

If you are interested in a more detailed description of how inversions work, tutorial 5 of this chapter collects the
full linear algebra in one place. The file
`autolens_workspace/*/imaging/features/pixelization/likelihood_function.ipynb` gives a complementary visual
step-by-step guide of the process alongside equations and references to literature on the subject.